in this notebook we will build and train ORM model 

Loads `microsoft/Phi-3-mini-4k-instruct` onto the GPU (`device_map="cuda"`, dtype chosen automatically) together with its tokenizer. This model is the frozen **backbone** of the ORM; we only need its hidden states, not its text generation

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer 

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
                "microsoft/Phi-3-mini-4k-instruct",
                device_map="cuda",
                torch_dtype="auto",
                trust_remote_code=True,
                )
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

initialize data 

so here we Loads four JSONL files into Hugging Face `DatasetDict`s: `test_orm`, `train_orm`, `val_orm`, and the full `data_orm` (which the other three were originally cut from). Each row has `question`, `solution`, and `label` (1 = correct, 0 = incorrect). Each loaded dict has a single `train` key, which is why later cells write `train_orm["train"]`, `val_orm["train"]`, etc.

In [ ]:
from datasets import load_dataset
#if you remeber from verifiers and build orm data notbook i have made a file called data_orm ,so test_orm is part from it i copy and pased data manually
test_orm = load_dataset(
    "json",
    data_files="test_orm.jsonl"
)

print(test_orm)

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['question', 'solution', 'label'],
        num_rows: 48
    })
})


In [73]:
from datasets import load_dataset

train_orm = load_dataset(
    "json",
    data_files="train_orm.jsonl"
)

print(train_orm)

DatasetDict({
    train: Dataset({
        features: ['question', 'solution', 'label'],
        num_rows: 300
    })
})


In [51]:
val_orm = load_dataset(
    "json",
    data_files="val_orm.jsonl"
)

print(val_orm)

DatasetDict({
    train: Dataset({
        features: ['question', 'solution', 'label'],
        num_rows: 52
    })
})


In [87]:
data = load_dataset(
    "json",
    data_files="data_orm.jsonl"
)

print(data)

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['question', 'solution', 'label'],
        num_rows: 400
    })
})


LOOK we will use LLM representation → correctness classification as an arch of our orm model 

which called 

(Feature Extraction / Frozen Backbone Fine-tuning) 

or
Frozen Pretrained Backbone + Trainable Classification Head for more clarifiy

or
Linear Probing




```
Pretrained Phi-3
      │ frozen
      ▼
Hidden States
      │
      ▼
Mean Pooling
      │
      ▼
Linear Classification Head  ← trainable
      │
      ▼
     Logit
      │
      ▼
     Loss
```

Quick sanity check of what was loaded
and i need to know model.config.hidden_size for the classifier head

In [ ]:
print(type(model))
print(model.config.hidden_size)##????
print(model.device)

<class 'transformers_modules.microsoft.Phi-3-mini-4k-instruct.f39ac1d28e925b323eae81227eaba4464caced4e.modeling_phi3.Phi3ForCausalLM'>
3072
cuda:0


ok before build the arch of model at once i need to check each part from arch work on his own first 
so i need to check apply chat template , tokenization , calc logit

In [ ]:
example = {'question': 'Brandon sold 86 geckos last year. He sold twice that many the year before. How many geckos has Brandon sold in the last two years?', 'solution': 'To solve this problem, we need to find out how many geckos Brandon sold the year before last year and then add that number to the 86 geckos he sold last year.\n\n1. Brandon sold 86 geckos last year.\n2. He sold twice that many the year before: 2 * 86 = 172 geckos.\n3. Add the geckos sold last year to the geckos sold the year before: 86 + 172 = 258 geckos.\n\nFinal Answer: 258', 'label': 1}
messages = [
    {
        "role": "user",
        "content": f"""Problem:
{example["question"]}

Candidate Solution:
{example["solution"]}

Determine whether the candidate solution is correct."""
    }
]

In [23]:
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=False
)

In [24]:
inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=4000
)

In [25]:
text

'<|user|>\nProblem:\nBrandon sold 86 geckos last year. He sold twice that many the year before. How many geckos has Brandon sold in the last two years?\n\nCandidate Solution:\nTo solve this problem, we need to find out how many geckos Brandon sold the year before last year and then add that number to the 86 geckos he sold last year.\n\n1. Brandon sold 86 geckos last year.\n2. He sold twice that many the year before: 2 * 86 = 172 geckos.\n3. Add the geckos sold last year to the geckos sold the year before: 86 + 172 = 258 geckos.\n\nFinal Answer: 258\n\nDetermine whether the candidate solution is correct.<|end|>\n<|endoftext|>'

In [ ]:
#This line moves all the tensors output by the tokenizer to the same device where the model is located.)(GPU)
inputs = {k: v.to(model.device) for k, v in inputs.items()}

In [28]:
c=[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
          1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
          1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
          1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
          1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
          1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
          1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
          1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
          1]
len(c)

193

In [27]:
inputs

{'input_ids': tensor([[32010, 11583, 29901,    13, 12432,  9214,  5239, 29871, 29947, 29953,
           1737,   384,   359,  1833,  1629, 29889,   940,  5239,  8951,   393,
           1784,   278,  1629,  1434, 29889,  1128,  1784,  1737,   384,   359,
            756,  1771,  9214,  5239,   297,   278,  1833,  1023,  2440, 29973,
             13,    13, 29907,  5380,   403, 24380, 29901,    13,  1762,  4505,
            445,  1108, 29892,   591,   817,   304,  1284,   714,   920,  1784,
           1737,   384,   359,  1771,  9214,  5239,   278,  1629,  1434,  1833,
           1629,   322,   769,   788,   393,  1353,   304,   278, 29871, 29947,
          29953,  1737,   384,   359,   540,  5239,  1833,  1629, 29889,    13,
             13, 29896, 29889,  1771,  9214,  5239, 29871, 29947, 29953,  1737,
            384,   359,  1833,  1629, 29889,    13, 29906, 29889,   940,  5239,
           8951,   393,  1784,   278,  1629,  1434, 29901, 29871, 29906,   334,
          29871, 29947, 299

ok now i will try to apply forward propagation on that input so i will get contextual representation and then apply linear classifer on this vector(contextual representation)

In [28]:
import torch

In [29]:
with torch.no_grad():
    outputs = model(
        **inputs,
        output_hidden_states=True
    )

You are not running the flash-attention implementation, expect numerical differences.


In [30]:
print("Number of hidden states:", len(outputs.hidden_states))
print("Shape of last hidden state:", outputs.hidden_states[-1].shape)

Number of hidden states: 33
Shape of last hidden state: torch.Size([1, 193, 3072])


so [1, 193, 3072] refer to that each token in the input represented by vector with 3072 dim 

Token 1  → vector [3072]

Token 2  → vector [3072]

Token 3  → vector [3072]

However, the ORM requires only a single vector that represents the question + the solution, so we will apply mean pooling

as we will make nn.Linear(3072, 1)

In [ ]:
hidden_states = outputs.hidden_states[-1]#contextual representation for the input
# look we made attention_mask because we don't need the padding vectors be in mean calculation
attention_mask = inputs["attention_mask"]# for idenify padding tokens and multiply it by zero for no callced in loss

mask = attention_mask.unsqueeze(-1).float()

masked_hidden = hidden_states * mask

pooled = masked_hidden.sum(dim=1) / mask.sum(dim=1) # عايزين نعبر عن ال كل التزكنز ف فيكتور واحد

print("Hidden states:", hidden_states.shape)
print("Pooled:", pooled.shape)

Hidden states: torch.Size([1, 193, 3072])
Pooled: torch.Size([1, 3072])


In [33]:
import torch.nn as nn

classifier = nn.Linear(3072, 1).to(model.device)

logit = classifier(pooled)

print("Logit shape:", logit.shape)
print("Logit:", logit)

Logit shape: torch.Size([1, 1])
Logit: tensor([[-0.0903]], device='cuda:0', grad_fn=<AddmmBackward0>)


now after we get logit(prediction) so let's ca;c the loss

In [34]:
label = torch.tensor(
    [[example["label"]]],
    dtype=torch.float32,
    device=model.device
)

print("Label:", label)

Label: tensor([[1.]], device='cuda:0')


In [35]:
loss_fn = nn.BCEWithLogitsLoss()

loss = loss_fn(logit, label)

print("Logit:", logit)
print("Label:", label)
print("Loss:", loss)

Logit: tensor([[-0.0903]], device='cuda:0', grad_fn=<AddmmBackward0>)
Label: tensor([[1.]], device='cuda:0')
Loss: tensor(0.7393, device='cuda:0',
       grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)


In [ ]:
probability = torch.sigmoid(logit)

print("Probability:", probability)#probabilty that solution is right as the label is 1

Probability: tensor([[0.4774]], device='cuda:0', grad_fn=<SigmoidBackward0>)


pipeline until now is 
```
Question + Solution
        ↓
    Tokenizer
        ↓
      Phi-3
        ↓
 Hidden States
        ↓
 Mean Pooling
        ↓
   [1, 3072]
        ↓
 Linear(3072 → 1)
        ↓
   logit = -0.0903
        ↓
BCEWithLogitsLoss
        ↑
 label = 1
```

Backpropagation and optimization(weights update) 

In [37]:
loss.backward()

In [38]:
print("Weight gradient shape:", classifier.weight.grad.shape)
print("Bias gradient:", classifier.bias.grad)

Weight gradient shape: torch.Size([1, 3072])
Bias gradient: tensor([-0.5226], device='cuda:0')


In [39]:
print("Before update:")
print("Logit:", logit.item())
print("Probability:", torch.sigmoid(logit).item())

Before update:
Logit: -0.0903279259800911
Probability: 0.4774334132671356


In [40]:
optimizer = torch.optim.AdamW(
    classifier.parameters(),
    lr=1e-3
)

In [41]:
optimizer.step()

In [42]:
optimizer.zero_grad()

In [ ]:
with torch.no_grad():# here forword with updated weights
    new_logit = classifier(pooled)
    new_probability = torch.sigmoid(new_logit)

print("After update:")
print("Logit:", new_logit.item())
print("Probability:", new_probability.item())

After update:
Logit: 1.4335030317306519
Probability: 0.8074465394020081


so The model shifted the prediction from:
47.7% → 80.7%

This is because the gradient indicated to the optimizer that the logit needed to move in the positive direction.

Now it's time to put everything together
in one class

In [ ]:
import torch
import torch.nn as nn

class ORMModel(nn.Module):

    def __init__(self, backbone):
        super().__init__()

        self.backbone = backbone#phi

        self.classifier = nn.Linear(
            backbone.config.hidden_size,
            1
        ).to(model.device)

    def forward(
        self,
        input_ids,
        attention_mask,
        labels=None
    ):

        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )

        hidden_states = outputs.hidden_states[-1]

        mask = attention_mask.unsqueeze(-1).float()

        masked_hidden = hidden_states * mask

        pooled = masked_hidden.sum(dim=1) / mask.sum(dim=1)

        logits = self.classifier(pooled)

        if labels is not None:

            loss_fn = nn.BCEWithLogitsLoss()

            loss = loss_fn(
                logits,
                labels
            )

            return {
                "loss": loss,
                "logits": logits
            }

        return {
            "logits": logits
        }

```
                    ORMModel
                       │
              ┌────────┴────────┐
              │                 │
          Phi-3 backbone     classifier
              │                 │
              ↓                 │
        hidden states            │
              ↓                 │
        mean pooling             │
              └──────→ Linear ──┘
                         ↓
                       logit
                         ↓
                  BCEWithLogitsLoss
```


Next step: Perform a forward pass and calculate the loss for a sample.
Before moving on to the DataLoader or full-scale training, we need to ensure that the entire ORMModel functions correctly end-to-end.

In [54]:
orm_model = ORMModel(model)

In [55]:
print(orm_model.classifier)
print(orm_model.classifier.weight.shape)

Linear(in_features=3072, out_features=1, bias=True)
torch.Size([1, 3072])


In [49]:
example = train_orm["train"][0]
example

{'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?',
 'solution': 'Step 1: Determine the number of clips sold in April.\nNatalia sold clips to 48 of her friends in April.\n\nStep 2: Determine the number of clips sold in May.\nShe sold half as many clips in May as in April, so she sold (1/2) * 48 = 24 clips in May.\n\nStep 3: Add the number of clips sold in April and May to find the total.\nTotal number of clips sold = 48 (April) + 24 (May) = 72.\n\nFinal Answer: 72',
 'label': 1}

In [56]:
example = train_orm["train"][0]

messages = [
    {
        "role": "user",
        "content": f"""Problem:
{example["question"]}

Candidate Solution:
{example["solution"]}

Determine whether the candidate solution is correct."""
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=False
)

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=2048
)

inputs = {
    k: v.to(model.device)
    for k, v in inputs.items()
}

labels = torch.tensor(
    [[example["label"]]],
    dtype=torch.float32,
    device=model.device
)

outputs = orm_model(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    labels=labels
)

print("Loss:", outputs["loss"])
print("Logit:", outputs["logits"])
print("Probability:", torch.sigmoid(outputs["logits"]))

Loss: tensor(0.5114, device='cuda:0',
       grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
Logit: tensor([[0.4041]], device='cuda:0', grad_fn=<AddmmBackward0>)
Probability: tensor([[0.5997]], device='cuda:0', grad_fn=<SigmoidBackward0>)


In [ ]:
for param in orm_model.backbone.parameters():
    param.requires_grad = False #to freeze phi layers

In [ ]:
trainable_params = sum(
    p.numel()
    for p in orm_model.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in orm_model.parameters()
)

print("Trainable parameters:", trainable_params)#for nn.linear(in classifier)
print("Total parameters:", total_params)

Trainable parameters: 3073
Total parameters: 3821082625


In [59]:
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, orm_model.parameters()),
    lr=1e-3
)

In [60]:
optimizer.zero_grad()

outputs = orm_model(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    labels=labels
)

loss = outputs["loss"]

print("Loss before backward:", loss.item())

loss.backward()

print("Classifier weight gradient:")
print(orm_model.classifier.weight.grad.shape)

print("Classifier bias gradient:")
print(orm_model.classifier.bias.grad)

optimizer.step()

Loss before backward: 0.5113762617111206
Classifier weight gradient:
torch.Size([1, 3072])
Classifier bias gradient:
tensor([-0.4003], device='cuda:0')


In [61]:
# Prediction before update
with torch.no_grad():
    before = orm_model(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"]
    )

    before_prob = torch.sigmoid(before["logits"])

print("Before optimizer step:")
print("Logit:", before["logits"].item())
print("Probability:", before_prob.item())


# Update
optimizer.step()
optimizer.zero_grad()


# Prediction after update
with torch.no_grad():
    after = orm_model(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"]
    )

    after_prob = torch.sigmoid(after["logits"])

print("\nAfter optimizer step:")
print("Logit:", after["logits"].item())
print("Probability:", after_prob.item())

Before optimizer step:
Logit: 1.9663782119750977
Probability: 0.8772215247154236

After optimizer step:
Logit: 3.5286519527435303
Probability: 0.9714921116828918


Excellent. We have now verified the entire pipeline using a single sample.

Note the difference:
Before update:
logit = 1.966
probability = 0.877

After update:
logit = 3.529
probability = 0.971

Since the label is 1, the optimizer has successfully moved the classifier in the right direction.

let's work on all dataset

In [ ]:
# for tokenize data
def preprocess(example):

    messages = [
        {
            "role": "user",
            "content": f"""Problem:
{example["question"]}

Candidate Solution:
{example["solution"]}

Determine whether the candidate solution is correct."""
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    encoded = tokenizer(
        text,
        truncation=True,
        max_length=2048
    )

    encoded["labels"] = float(example["label"])

    return encoded

In [63]:
tokenized_dataset = train_orm["train"].map(
    preprocess,
    remove_columns=train_orm["train"].column_names
)

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

In [64]:
print(tokenized_dataset)
print(tokenized_dataset[0])

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 300
})
{'input_ids': [32010, 11583, 29901, 13, 29940, 2075, 423, 5239, 9335, 567, 304, 29871, 29946, 29947, 310, 902, 7875, 297, 3786, 29892, 322, 769, 1183, 5239, 4203, 408, 1784, 9335, 567, 297, 2610, 29889, 1128, 1784, 9335, 567, 1258, 26259, 423, 19417, 19148, 297, 3786, 322, 2610, 29973, 13, 13, 29907, 5380, 403, 24380, 29901, 13, 14448, 29871, 29896, 29901, 5953, 837, 457, 278, 1353, 310, 9335, 567, 5239, 297, 3786, 29889, 13, 29940, 2075, 423, 5239, 9335, 567, 304, 29871, 29946, 29947, 310, 902, 7875, 297, 3786, 29889, 13, 13, 14448, 29871, 29906, 29901, 5953, 837, 457, 278, 1353, 310, 9335, 567, 5239, 297, 2610, 29889, 13, 13468, 5239, 4203, 408, 1784, 9335, 567, 297, 2610, 408, 297, 3786, 29892, 577, 1183, 5239, 313, 29896, 29914, 29906, 29897, 334, 29871, 29946, 29947, 353, 29871, 29906, 29946, 9335, 567, 297, 2610, 29889, 13, 13, 14448, 29871, 29941, 29901, 3462, 278, 1353, 310, 9335, 567, 5239,

In [65]:
from transformers import DataCollatorWithPadding
from torch.utils.data import DataLoader

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    return_tensors="pt"
)

train_loader = DataLoader(
    tokenized_dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=data_collator
)

print("Number of batches:", len(train_loader))

Number of batches: 75


In [66]:
batch = next(iter(train_loader))

print("input_ids:", batch["input_ids"].shape)
print("attention_mask:", batch["attention_mask"].shape)
print("labels:", batch["labels"].shape)
print("labels:", batch["labels"])

input_ids: torch.Size([4, 318])
attention_mask: torch.Size([4, 318])
labels: torch.Size([4])
labels: tensor([1., 0., 0., 1.])


In [ ]:
orm_model.train()

total_loss = 0

for step, batch in enumerate(train_loader):

    batch = {
        k: v.to(model.device)
        for k, v in batch.items()
    }

    optimizer.zero_grad()

    outputs = orm_model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        labels=batch["labels"].unsqueeze(1)
    )

    loss = outputs["loss"]

    loss.backward()

    optimizer.step()

    total_loss += loss.item()

    if step % 10 == 0:
        print(
            f"Step {step}/{len(train_loader)} "
            f"Loss: {loss.item():.4f}"
        )

avg_loss = total_loss / len(train_loader)

print("Average training loss:", avg_loss)

Step 0/75 Loss: 1.2422
Step 10/75 Loss: 0.0164
Step 20/75 Loss: 0.9708
Step 30/75 Loss: 0.2227
Step 40/75 Loss: 0.3821
Step 50/75 Loss: 0.2661
Step 60/75 Loss: 0.2222
Step 70/75 Loss: 0.2514
Average training loss: 0.5222713363046447


Next step: Validation


In [69]:
val_tokenized = val_orm["train"].map(
    preprocess,
    remove_columns=val_orm["train"].column_names
)

val_loader = DataLoader(
    val_tokenized,
    batch_size=4,
    shuffle=False,
    collate_fn=data_collator
)

print("Validation batches:", len(val_loader))

Map:   0%|          | 0/52 [00:00<?, ? examples/s]

Validation batches: 13


In [70]:
orm_model.eval()

total_val_loss = 0
correct = 0
total = 0

with torch.no_grad():

    for batch in val_loader:

        batch = {
            k: v.to(model.device)
            for k, v in batch.items()
        }

        outputs = orm_model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"]
        )

        logits = outputs["logits"]

        probabilities = torch.sigmoid(logits)

        predictions = (probabilities >= 0.5).float()

        labels = batch["labels"].unsqueeze(1)

        loss_fn = nn.BCEWithLogitsLoss()

        loss = loss_fn(
            logits,
            labels
        )

        total_val_loss += loss.item()

        correct += (predictions == labels).sum().item()
        total += labels.numel()

val_loss = total_val_loss / len(val_loader)
val_accuracy = correct / total

print("Validation Loss:", val_loss)
print("Validation Accuracy:", val_accuracy)

Validation Loss: 0.27127605218153733
Validation Accuracy: 0.9615384615384616


However, there is an important point: 96.15% does not necessarily mean that the ORM is excellent—especially considering that validation is based on only 50 samples.

Next step: Let’s examine the ORM’s performance in detail.
We want to determine whether the model can effectively distinguish between:
label = 1 → solution correct
label = 0 → solution incorrect

In [71]:
from sklearn.metrics import confusion_matrix, classification_report

orm_model.eval()

all_predictions = []
all_labels = []
all_probabilities = []

with torch.no_grad():

    for batch in val_loader:

        batch = {
            k: v.to(model.device)
            for k, v in batch.items()
        }

        outputs = orm_model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"]
        )

        probabilities = torch.sigmoid(outputs["logits"])

        predictions = (probabilities >= 0.5).long()

        all_probabilities.extend(
            probabilities.squeeze(1).cpu().tolist()
        )

        all_predictions.extend(
            predictions.squeeze(1).cpu().tolist()
        )

        all_labels.extend(
            batch["labels"].long().cpu().tolist()
        )


print(
    confusion_matrix(
        all_labels,
        all_predictions
    )
)

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=["Incorrect", "Correct"]
    )
)

[[ 0  1]
 [ 1 50]]
              precision    recall  f1-score   support

   Incorrect       0.00      0.00      0.00         1
     Correct       0.98      0.98      0.98        51

    accuracy                           0.96        52
   macro avg       0.49      0.49      0.49        52
weighted avg       0.96      0.96      0.96        52



**A very important point emerged here before we moved on to the test phase.**

The 96% accuracy was misleading due to the distribution of the validation set.

Actual Incorrect: 1.

Actual Correct:   51.

Total:            52

so from the confusion_matrix we get
TN = 0
FP = 1
FN = 1
TP = 50
so model doesn't generate (this is a wrong answer)

**so We have a very severe class imbalance: 51 correct vs. 1 incorrect. Consequently,** 

**we cannot consider 96% accuracy as strong evidence that the ORM has learned to identify incorrect solutions.**

First, let’s check the splits and the labels

In [75]:
print("Train size:", len(train_orm["train"]))
print("Val size:", len(val_orm["train"]))
print("Test size:", len(test_orm["train"]))

print("\nTrain labels:")
print(train_orm["train"]["label"].count(0),
      train_orm["train"]["label"].count(1))

print("\nVal labels:")
print(val_orm["train"]["label"].count(0),
      val_orm["train"]["label"].count(1))

print("\nTest labels:")
print(test_orm["train"]["label"].count(0),
      test_orm["train"]["label"].count(1))

Train size: 300
Val size: 52
Test size: 48

Train labels:
67 233

Val labels:
1 51

Test labels:
9 39


so the imblance between labels is too big also 

The distribution of labels across the groups(train,val,test) is imbalanced.

In [76]:
from collections import Counter

questions = train_orm["train"]["question"]

counts = Counter(questions)

print("Number of unique questions:", len(counts))
print("Solutions per question:")

print(Counter(counts.values()))

Number of unique questions: 75
Solutions per question:
Counter({4: 75})


In [ ]:
# confirm that no data leakage
train_questions = set(train_orm["train"]["question"])
val_questions = set(val_orm["train"]["question"])
test_questions = set(test_orm["train"]["question"])

print("Train questions:", len(train_questions))
print("Val questions:", len(val_questions))
print("Test questions:", len(test_questions))

print("\nTrain ∩ Val:", len(train_questions & val_questions))
print("Train ∩ Test:", len(train_questions & test_questions))
print("Val ∩ Test:", len(val_questions & test_questions))

Train questions: 75
Val questions: 13
Test questions: 12

Train ∩ Val: 0
Train ∩ Test: 0
Val ∩ Test: 0


In [ ]:
from collections import Counter

def question_label_distribution(split):

    questions = {}

    for item in split:
        q = item["question"]

        if q not in questions:
            questions[q] = []

        questions[q].append(item["label"])

    for q, labels in questions.items():
        print(
            "labels:",
            labels,
            "incorrect:",
            labels.count(0),
            "correct:",
            labels.count(1)
        )

    print("\nQuestion-level distribution:")

    counter = Counter(
        tuple(labels)
        for labels in questions.values()
    )

    print(counter)


question_label_distribution(val_orm["train"])#to understand the distribution of labels at the question level.

labels: [1, 1, 0, 1] incorrect: 1 correct: 3
labels: [1, 1, 1, 1] incorrect: 0 correct: 4
labels: [1, 1, 1, 1] incorrect: 0 correct: 4
labels: [1, 1, 1, 1] incorrect: 0 correct: 4
labels: [1, 1, 1, 1] incorrect: 0 correct: 4
labels: [1, 1, 1, 1] incorrect: 0 correct: 4
labels: [1, 1, 1, 1] incorrect: 0 correct: 4
labels: [1, 1, 1, 1] incorrect: 0 correct: 4
labels: [1, 1, 1, 1] incorrect: 0 correct: 4
labels: [1, 1, 1, 1] incorrect: 0 correct: 4
labels: [1, 1, 1, 1] incorrect: 0 correct: 4
labels: [1, 1, 1, 1] incorrect: 0 correct: 4
labels: [1, 1, 1, 1] incorrect: 0 correct: 4

Question-level distribution:
Counter({(1, 1, 1, 1): 12, (1, 1, 0, 1): 1})


look now as we know that the train have 67 incorrect sol and val only have one so that a big imbalance so we need to balance but note we don't need to fall in data leakage so we don't need to split by solutions only we also have to consider questions

In [80]:
def question_distribution(split):

    questions = {}

    for item in split:
        q = item["question"]

        if q not in questions:
            questions[q] = []

        questions[q].append(item["label"])

    return questions


for name, split in [
    ("Train", train_orm["train"]),
    ("Val", val_orm["train"]),
    ("Test", test_orm["train"])
]:

    questions = question_distribution(split)

    print(f"\n{name}")

    print("Questions:", len(questions))

    print(
        "Questions containing incorrect solution:",
        sum(
            0 in labels
            for labels in questions.values()
        )
    )

    print(
        "Questions with all correct:",
        sum(
            all(label == 1 for label in labels)
            for labels in questions.values()
        )
    )


Train
Questions: 75
Questions containing incorrect solution: 40
Questions with all correct: 35

Val
Questions: 13
Questions containing incorrect solution: 1
Questions with all correct: 12

Test
Questions: 12
Questions containing incorrect solution: 5
Questions with all correct: 7


In [88]:
from collections import defaultdict

data = data["train"]

groups = defaultdict(list)

for i, row in enumerate(data):
    groups[row["question"]].append(i)

print("Number of unique questions:", len(groups))

print("Solutions per question:")
print(set(len(indices) for indices in groups.values()))

Number of unique questions: 100
Solutions per question:
{4}


In [89]:
question_stats = []

for question, indices in groups.items():

    labels = [data[i]["label"] for i in indices]

    num_incorrect = labels.count(0)
    num_correct = labels.count(1)

    question_stats.append({
        "question": question,
        "num_incorrect": num_incorrect,
        "num_correct": num_correct
    })

from collections import Counter

distribution = Counter(
    x["num_incorrect"]
    for x in question_stats
)

print(distribution)

Counter({0: 54, 1: 24, 2: 15, 3: 5, 4: 2})


```
54 questions → 0 incorrect → [1,1,1,1]
24 questions → 1 incorrect → [0,1,1,1]
15 questions → 2 incorrect → [0,0,1,1]
 5 questions → 3 incorrect → [0,0,0,1]
 2 questions → 4 incorrect → [0,0,0,0]
--------------------------------
100 questions
```

For example, we can use a distribution:
```
                    0-inc  1-inc  2-inc  3-inc  4-inc
Train (75)            40     18     13      3      2
Validation (13)        7      3      1      1      1
Test (12)              7      3      1      1      0

This is excellent because the validation and test sets will actually contain incorrect solutions, unlike the previous issue
```

In [92]:
import random

# Convert question_stats into groups by number of incorrect solutions
strata = {
    0: [],
    1: [],
    2: [],
    3: [],
    4: []
}

for item in question_stats:
    strata[item["num_incorrect"]].append(item["question"])


# Fixed seed for reproducibility
random.seed(42)

for k in strata:
    random.shuffle(strata[k])


# Number of questions from each stratum
train_counts = {
    0: 41,
    1: 18,
    2: 11,
    3: 4,
    4: 1
}

val_counts = {
    0: 7,
    1: 3,
    2: 1,
    3: 0,
    4: 1
}

test_counts = {
    0: 6,
    1: 3,
    2: 3,
    3: 1,
    4: 0
}

train_questions = []
val_questions = []
test_questions = []

for k in strata:

    questions = strata[k]

    n_train = train_counts[k]
    n_val = val_counts[k]
    n_test = test_counts[k]

    train_questions.extend(
        questions[:n_train]
    )

    val_questions.extend(
        questions[n_train:n_train + n_val]
    )

    test_questions.extend(
        questions[n_train + n_val:n_train + n_val + n_test]
    )


print("Train questions:", len(train_questions))
print("Val questions:", len(val_questions))
print("Test questions:", len(test_questions))

Train questions: 75
Val questions: 12
Test questions: 13


In [93]:
def get_indices(question_list):
    indices = []

    for question in question_list:
        indices.extend(groups[question])

    return indices


train_indices = get_indices(train_questions)
val_indices = get_indices(val_questions)
test_indices = get_indices(test_questions)


print("Train rows:", len(train_indices))
print("Val rows:", len(val_indices))
print("Test rows:", len(test_indices))

Train rows: 300
Val rows: 48
Test rows: 52


In [ ]:
train_dataset = data.select(train_indices)
val_dataset = data.select(val_indices)
test_dataset = data.select(test_indices)

In [95]:
print(train_dataset)
print(val_dataset)
print(test_dataset)

Dataset({
    features: ['question', 'solution', 'label'],
    num_rows: 300
})
Dataset({
    features: ['question', 'solution', 'label'],
    num_rows: 48
})
Dataset({
    features: ['question', 'solution', 'label'],
    num_rows: 52
})


In [3]:
from datasets import load_from_disk

train_dataset = load_from_disk("orm_train")
val_dataset = load_from_disk("orm_val")
test_dataset = load_from_disk("orm_test")
print(train_dataset)
print(val_dataset)
print(test_dataset)

Dataset({
    features: ['question', 'solution', 'label'],
    num_rows: 300
})
Dataset({
    features: ['question', 'solution', 'label'],
    num_rows: 48
})
Dataset({
    features: ['question', 'solution', 'label'],
    num_rows: 52
})


In [96]:
from collections import Counter

print("Train labels:", Counter(train_dataset["label"]))
print("Val labels:", Counter(val_dataset["label"]))
print("Test labels:", Counter(test_dataset["label"]))

Train labels: Counter({1: 244, 0: 56})
Val labels: Counter({1: 39, 0: 9})
Test labels: Counter({1: 40, 0: 12})


```
| Split | Questions | Rows | Correct | Incorrect |
|-------|-----------|------|---------|-----------|
| Train | 75        | 300  | 244     | 56        |
| Val   | 12        | 48   | 39      | 9         |
| Test  | 13        | 52   | 40      | 12        |

```

In [ ]:
#We ensure that there is no question leakage
train_q = set(train_dataset["question"])
val_q = set(val_dataset["question"])
test_q = set(test_dataset["question"])

print("Train questions:", len(train_q))
print("Val questions:", len(val_q))
print("Test questions:", len(test_q))

print("Train ∩ Val:", len(train_q & val_q))
print("Train ∩ Test:", len(train_q & test_q))
print("Val ∩ Test:", len(val_q & test_q))

Train questions: 75
Val questions: 12
Test questions: 13
Train ∩ Val: 0
Train ∩ Test: 0
Val ∩ Test: 0


after we balanced data let's train on it

In [6]:
train_tokenized = train_dataset.map(
    preprocess,
    remove_columns=train_dataset.column_names
)

val_tokenized = val_dataset.map(
    preprocess,
    remove_columns=val_dataset.column_names
)

test_tokenized = test_dataset.map(
    preprocess,
    remove_columns=test_dataset.column_names
)

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Map:   0%|          | 0/48 [00:00<?, ? examples/s]

Map:   0%|          | 0/52 [00:00<?, ? examples/s]

In [26]:
train_tokenized

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 300
})

In [7]:
from transformers import DataCollatorWithPadding
from torch.utils.data import DataLoader

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    return_tensors="pt"
)

train_loader = DataLoader(
    train_tokenized,
    batch_size=4,
    shuffle=True,
    collate_fn=data_collator
)

val_loader = DataLoader(
    val_tokenized,
    batch_size=4,
    shuffle=False,
    collate_fn=data_collator
)

test_loader = DataLoader(
    test_tokenized,
    batch_size=4,
    shuffle=False,
    collate_fn=data_collator
)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

Train batches: 75
Val batches: 12
Test batches: 13


test

In [100]:
batch = next(iter(train_loader))

print("input_ids:", batch["input_ids"].shape)
print("attention_mask:", batch["attention_mask"].shape)
print("labels:", batch["labels"].shape)
print("labels:", batch["labels"])

input_ids: torch.Size([4, 368])
attention_mask: torch.Size([4, 368])
labels: torch.Size([4])
labels: tensor([1., 1., 1., 1.])


In [101]:
orm_model = ORMModel(model)

for param in orm_model.backbone.parameters():
    param.requires_grad = False

orm_model = orm_model.to(model.device)

In [102]:
trainable_params = sum(
    p.numel()
    for p in orm_model.parameters()
    if p.requires_grad
)

print("Trainable parameters:", trainable_params)

Trainable parameters: 3073


In [103]:
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, orm_model.parameters()),
    lr=1e-3
)

In [104]:
orm_model.train()

total_loss = 0

for step, batch in enumerate(train_loader):

    batch = {
        k: v.to(model.device)
        for k, v in batch.items()
    }

    optimizer.zero_grad()

    outputs = orm_model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        labels=batch["labels"].unsqueeze(1)
    )

    loss = outputs["loss"]

    loss.backward()

    optimizer.step()

    total_loss += loss.item()

    if step % 10 == 0:
        print(
            f"Step {step}/{len(train_loader)} "
            f"Loss: {loss.item():.4f}"
        )

avg_train_loss = total_loss / len(train_loader)

print("Average training loss:", avg_train_loss)

Step 0/75 Loss: 0.6843
Step 10/75 Loss: 0.3907
Step 20/75 Loss: 1.0091
Step 30/75 Loss: 0.7368
Step 40/75 Loss: 0.1199
Step 50/75 Loss: 0.6283
Step 60/75 Loss: 0.6315
Step 70/75 Loss: 0.5904
Average training loss: 0.4993999720613162


In [105]:
import torch

orm_model.eval()

total_loss = 0
correct = 0
total = 0

all_preds = []
all_labels = []

with torch.no_grad():

    for batch in val_loader:

        batch = {
            k: v.to(model.device)
            for k, v in batch.items()
        }

        outputs = orm_model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"].unsqueeze(1)
        )

        loss = outputs["loss"]
        logits = outputs["logits"]

        probabilities = torch.sigmoid(logits)

        preds = (probabilities >= 0.5).float()

        total_loss += loss.item()

        correct += (
            preds == batch["labels"].unsqueeze(1)
        ).sum().item()

        total += batch["labels"].size(0)

        all_preds.extend(
            preds.squeeze(1).cpu().tolist()
        )

        all_labels.extend(
            batch["labels"].cpu().tolist()
        )


val_loss = total_loss / len(val_loader)
val_accuracy = correct / total

print("Validation Loss:", val_loss)
print("Validation Accuracy:", val_accuracy)

Validation Loss: 0.3722321940585971
Validation Accuracy: 0.8125


In [106]:
from sklearn.metrics import confusion_matrix, classification_report

print(confusion_matrix(all_labels, all_preds))

print(
    classification_report(
        all_labels,
        all_preds,
        target_names=["Incorrect", "Correct"],
        zero_division=0
    )
)

[[ 0  9]
 [ 0 39]]
              precision    recall  f1-score   support

   Incorrect       0.00      0.00      0.00         9
     Correct       0.81      1.00      0.90        39

    accuracy                           0.81        48
   macro avg       0.41      0.50      0.45        48
weighted avg       0.66      0.81      0.73        48



**as you see TN = 0 and FP = 9 it's mean that model still doesn't learn to say this a wrong answer**

save model and data

In [107]:
import torch

checkpoint = {
    "model_state_dict": orm_model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
}

torch.save(
    checkpoint,
    "orm_checkpoint.pt"
)

print("Checkpoint saved.")

Checkpoint saved.


In [ ]:
train_dataset.save_to_disk("orm_train")#as you see i call it above using load_from_disk
val_dataset.save_to_disk("orm_val")
test_dataset.save_to_disk("orm_test")

print("Datasets saved.")

Saving the dataset (0/1 shards):   0%|          | 0/300 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/48 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/52 [00:00<?, ? examples/s]

Datasets saved.


so now we need a sol for that imbalance in labels so we wil try to use class-weighted loss

explanition for class-weighted loss :Make mistakes on incorrect solutions count about 4.4× more in the loss (244 / 56). This is done with per-sample weights, not pos_weight, because pos_weight boosts the Correct class and we want the opposite.
so Compute the loss per example (reduction="none"), multiply each by its weight (4.4 for label 0, 1 for label 1), then take the mean.

The model was simply ignoring the minority class.

After applying weighting, an error—specifically misclassifying an "Incorrect" instance as "Correct"—

becomes approximately 4.36 times more costly than a standard error.

Consequently, the model gains a genuine incentive to learn how to identify "Incorrect" cases.

However, this does not necessarily mean that accuracy will increase. For instance, accuracy might drop from 81% to 75%, yet the model’s performance (in terms of the metric of interest) could be significantly better, because the recall for the "Incorrect" class is no longer zero.

In [9]:
import torch
import torch.nn as nn
class ORMModel(nn.Module):

    def __init__(self, backbone):
        super().__init__()

        self.backbone = backbone

        self.classifier = nn.Linear(
            backbone.config.hidden_size,
            1
        )

    def forward(
        self,
        input_ids,
        attention_mask,
        labels=None
    ):

        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )

        hidden_states = outputs.hidden_states[-1]

        mask = attention_mask.unsqueeze(-1).float()

        masked_hidden = hidden_states * mask

        pooled = masked_hidden.sum(dim=1) / mask.sum(dim=1)

        logits = self.classifier(pooled)

        if labels is not None:

            loss_fn = nn.BCEWithLogitsLoss(
                reduction="none"
            )

            losses = loss_fn(
                logits,
                labels
            )

            weights = torch.where(
                labels == 0,
                torch.tensor(
                    244 / 56,
                    device=labels.device
                ),
                torch.tensor(
                    1.0,
                    device=labels.device
                )
            )

            loss = (losses * weights).mean()

            return {
                "loss": loss,
                "logits": logits
            }

        return {
            "logits": logits
        }

In [13]:
orm_model = ORMModel(model)

for param in orm_model.backbone.parameters():
    param.requires_grad = False

orm_model = orm_model.to(model.device)

In [ ]:
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, orm_model.parameters()),
    lr=1e-3
)

In [15]:
print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in orm_model.parameters()
        if p.requires_grad
    )
)

Trainable parameters: 3073


In [16]:
orm_model.train()
total_loss=0
for step,batch in enumerate(train_loader):
    batch={
        k: v.to(model.device)
        for k, v in batch.items()
    }
    optimizer.zero_grad()
    outputs = orm_model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"].unsqueeze(1)
        )
    loss = outputs["loss"]
    
    loss.backward()

    optimizer.step()

    total_loss += loss.item()

    if step % 10 == 0:
        print(
            f"Step {step}/{len(train_loader)} "
            f"Loss: {loss.item():.4f}"
        )

avg_loss = total_loss / len(train_loader)

print("Average training loss:", avg_loss)

You are not running the flash-attention implementation, expect numerical differences.


Step 0/75 Loss: 0.6277
Step 10/75 Loss: 1.3642
Step 20/75 Loss: 1.2339
Step 30/75 Loss: 0.1314
Step 40/75 Loss: 1.5180
Step 50/75 Loss: 1.2991
Step 60/75 Loss: 2.1272
Step 70/75 Loss: 1.2003
Average training loss: 1.526984892586867


Yes, this result is entirely expected after applying Weighted BCE; it does not mean the training has failed.

Note the difference:
Before weighting:
Average training loss = 0.499

After weighting:
Average training loss = 1.527

The loss has increased, which is natural since we have altered the definition of the loss function itself.
Why has the loss become high and volatile?
as we have set:
Incorrect → weight = 4.36
Correct → weight = 1.0

Consequently, if a batch contains "incorrect" samples—those the model is still misclassifying—the loss can spike significantly.

In [17]:
orm_model.eval()

total_loss = 0
correct = 0
total = 0

all_preds = []
all_labels = []

with torch.no_grad():

    for batch in val_loader:

        batch = {
            k: v.to(model.device)
            for k, v in batch.items()
        }

        outputs = orm_model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"].unsqueeze(1)
        )

        loss = outputs["loss"]
        logits = outputs["logits"]

        probabilities = torch.sigmoid(logits)

        preds = (probabilities >= 0.5).float()

        total_loss += loss.item()

        correct += (
            preds == batch["labels"].unsqueeze(1)
        ).sum().item()

        total += batch["labels"].size(0)

        all_preds.extend(
            preds.squeeze(1).cpu().tolist()
        )

        all_labels.extend(
            batch["labels"].cpu().tolist()
        )

val_loss = total_loss / len(val_loader)
val_accuracy = correct / total

print("Validation Loss:", val_loss)
print("Validation Accuracy:", val_accuracy)

Validation Loss: 0.8887641355395317
Validation Accuracy: 0.8541666666666666


In [19]:
from sklearn.metrics import confusion_matrix, classification_report
print(
    confusion_matrix(
        all_labels,
        all_preds
    )
)

print(
    classification_report(
        all_labels,
        all_preds,
        target_names=["Incorrect", "Correct"]
    )
)

[[ 5  4]
 [ 3 36]]
              precision    recall  f1-score   support

   Incorrect       0.62      0.56      0.59         9
     Correct       0.90      0.92      0.91        39

    accuracy                           0.85        48
   macro avg       0.76      0.74      0.75        48
weighted avg       0.85      0.85      0.85        48



**Key Improvement**
```
Before:
Incorrect:
9 samples → The model classified all of them as "Correct"

Meaning:
Recall = 0%

Now:
9 Incorrect
↓
5 correctly identified
4 misclassified

So:
Incorrect Recall = 5 / 9 = 55.6%

This is a significant improvement.

At the same time, we haven't compromised the performance on the "Correct" samples:
Correct:
36 / 39

Meaning:
Correct Recall = 92.3%
```

In [20]:
torch.save(
    {
        "model_state_dict": orm_model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
    },
    "orm_weighted_bce_checkpoint.pt"
)

print("Weighted BCE checkpoint saved.")

Weighted BCE checkpoint saved.


**We have a very important step to take:**
```
Conducting a fair comparison between different thresholds.
Currently, we have been using:
probability >= 0.5
```

However, since this involves an ORM (Outcome-Reward Model) verifier, a different threshold might yield better results—for instance:
```
0.3
0.4
0.5
0.6
0.7
```

We will then examine the impact of these thresholds on the "Incorrect F1" and "Macro F1" scores.

In [21]:
import torch
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

orm_model.eval()

all_probs = []
all_labels = []

with torch.no_grad():

    for batch in val_loader:

        batch = {
            k: v.to(model.device)
            for k, v in batch.items()
        }

        outputs = orm_model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"]
        )

        probs = torch.sigmoid(outputs["logits"])

        all_probs.extend(
            probs.squeeze(1).cpu().tolist()
        )

        all_labels.extend(
            batch["labels"].cpu().tolist()
        )


# Threshold analysis
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

print(
    f"{'Threshold':<10}"
    f"{'Accuracy':<10}"
    f"{'Inc Prec':<10}"
    f"{'Inc Recall':<12}"
    f"{'Inc F1':<10}"
    f"{'Macro F1':<10}"
)

for threshold in thresholds:

    preds = [
        1 if prob >= threshold else 0
        for prob in all_probs
    ]

    accuracy = accuracy_score(
        all_labels,
        preds
    )

    incorrect_precision = precision_score(
        all_labels,
        preds,
        pos_label=0,
        zero_division=0
    )

    incorrect_recall = recall_score(
        all_labels,
        preds,
        pos_label=0,
        zero_division=0
    )

    incorrect_f1 = f1_score(
        all_labels,
        preds,
        pos_label=0,
        zero_division=0
    )

    macro_f1 = f1_score(
        all_labels,
        preds,
        average="macro",
        zero_division=0
    )

    print(
        f"{threshold:<10.1f}"
        f"{accuracy:<10.3f}"
        f"{incorrect_precision:<10.3f}"
        f"{incorrect_recall:<12.3f}"
        f"{incorrect_f1:<10.3f}"
        f"{macro_f1:<10.3f}"
    )

Threshold Accuracy  Inc Prec  Inc Recall  Inc F1    Macro F1  
0.1       0.812     0.000     0.000       0.000     0.448     
0.2       0.812     0.000     0.000       0.000     0.448     
0.3       0.854     0.750     0.333       0.462     0.689     
0.4       0.875     0.714     0.556       0.625     0.775     
0.5       0.854     0.625     0.556       0.588     0.750     
0.6       0.542     0.290     1.000       0.450     0.529     
0.7       0.271     0.205     1.000       0.340     0.263     
0.8       0.188     0.188     1.000       0.316     0.158     
0.9       0.188     0.188     1.000       0.316     0.158     


```
Experiment 1 — Normal BCE
Accuracy       = 81.2%
Incorrect F1   = 0%
Macro F1       = 44.8%

Experiment 2 — Weighted BCE
Threshold 0.5
Accuracy       = 85.4%
Incorrect F1   = 58.8%
Macro F1       = 75.0%

Experiment 2 — Weighted BCE
Threshold 0.4
Accuracy       = 87.5%
Incorrect F1   = 62.5%
Macro F1       = 77.5%
```

test

In [ ]:


import torch
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score
)

orm_model.eval()

threshold = 0.4 # as it is the one who give us the highest statistics

all_test_probs = []
all_test_labels = []

with torch.no_grad():

    for batch in test_loader:

        batch = {
            k: v.to(model.device)
            for k, v in batch.items()
        }

        outputs = orm_model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"]
        )

        probabilities = torch.sigmoid(
            outputs["logits"]
        )

        all_test_probs.extend(
            probabilities.squeeze(1).cpu().tolist()
        )

        all_test_labels.extend(
            batch["labels"].cpu().tolist()
        )


# Apply selected threshold
all_test_preds = [
    1 if prob >= threshold else 0
    for prob in all_test_probs
]


# Accuracy
test_accuracy = accuracy_score(
    all_test_labels,
    all_test_preds
)

print("Test Accuracy:", test_accuracy)


# Confusion Matrix
cm = confusion_matrix(
    all_test_labels,
    all_test_preds
)

print("\nConfusion Matrix:")
print(cm)


# Classification Report
print("\nClassification Report:")

print(
    classification_report(
        all_test_labels,
        all_test_preds,
        target_names=[
            "Incorrect",
            "Correct"
        ],
        zero_division=0
    )
)

Test Accuracy: 0.7884615384615384

Confusion Matrix:
[[ 2 10]
 [ 1 39]]

Classification Report:
              precision    recall  f1-score   support

   Incorrect       0.67      0.17      0.27        12
     Correct       0.80      0.97      0.88        40

    accuracy                           0.79        52
   macro avg       0.73      0.57      0.57        52
weighted avg       0.77      0.79      0.74        52



**Ok. Now, let’s evaluate the ORM on the very task it was built for: out of four possible solutions to a given problem, does it select the correct one?**

In [23]:
import torch
from collections import defaultdict

orm_model.eval()

# Group test samples by question
test_groups = defaultdict(list)

for row in test_dataset:
    test_groups[row["question"]].append(row)


results = []

with torch.no_grad():

    for question, candidates in test_groups.items():

        scores = []

        for candidate in candidates:

            messages = [
                {
                    "role": "user",
                    "content": f"""Problem:
{question}

Candidate Solution:
{candidate["solution"]}

Determine whether the candidate solution is correct."""
                }
            ]

            text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False
            )

            inputs = tokenizer(
                text,
                return_tensors="pt",
                truncation=True,
                max_length=2048
            )

            inputs = {
                k: v.to(model.device)
                for k, v in inputs.items()
            }

            outputs = orm_model(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"]
            )

            probability = torch.sigmoid(
                outputs["logits"]
            ).item()

            scores.append(probability)


        # ORM chooses the highest-scoring solution
        selected_idx = max(
            range(len(scores)),
            key=lambda i: scores[i]
        )

        selected_candidate = candidates[selected_idx]

        results.append({
            "question": question,
            "scores": scores,
            "labels": [
                candidate["label"]
                for candidate in candidates
            ],
            "selected_idx": selected_idx,
            "selected_label": selected_candidate["label"]
        })

In [24]:
num_questions = len(results)

successful_selections = sum(
    result["selected_label"] == 1
    for result in results
)

best_of_4_accuracy = (
    successful_selections / num_questions
)

print(
    f"Best-of-4 Accuracy: "
    f"{best_of_4_accuracy:.4f}"
)

print(
    f"Successful selections: "
    f"{successful_selections}/{num_questions}"
)

Best-of-4 Accuracy: 1.0000
Successful selections: 13/13


In [25]:
for i, result in enumerate(results):

    print(f"\nQuestion {i + 1}")

    print("Scores:", [
        round(score, 3)
        for score in result["scores"]
    ])

    print("Labels:", result["labels"])

    print(
        "Selected:",
        result["selected_idx"]
    )

    print(
        "Selected label:",
        result["selected_label"]
    )


Question 1
Scores: [0.589, 0.59, 0.572, 0.365]
Labels: [1, 1, 1, 1]
Selected: 1
Selected label: 1

Question 2
Scores: [0.686, 0.692, 0.688, 0.634]
Labels: [1, 1, 1, 1]
Selected: 1
Selected label: 1

Question 3
Scores: [0.583, 0.615, 0.564, 0.613]
Labels: [1, 1, 1, 1]
Selected: 1
Selected label: 1

Question 4
Scores: [0.563, 0.557, 0.566, 0.593]
Labels: [1, 1, 1, 1]
Selected: 3
Selected label: 1

Question 5
Scores: [0.634, 0.633, 0.646, 0.643]
Labels: [1, 1, 1, 1]
Selected: 2
Selected label: 1

Question 6
Scores: [0.538, 0.555, 0.558, 0.572]
Labels: [1, 1, 1, 1]
Selected: 3
Selected label: 1

Question 7
Scores: [0.688, 0.64, 0.665, 0.646]
Labels: [1, 1, 0, 1]
Selected: 0
Selected label: 1

Question 8
Scores: [0.681, 0.66, 0.641, 0.633]
Labels: [1, 1, 0, 1]
Selected: 0
Selected label: 1

Question 9
Scores: [0.577, 0.559, 0.588, 0.6]
Labels: [1, 0, 1, 1]
Selected: 3
Selected label: 1

Question 10
Scores: [0.529, 0.579, 0.546, 0.514]
Labels: [1, 1, 0, 0]
Selected: 1
Selected label: 1

Que

A key observation:
We have an interesting result:
Classification:
Test Accuracy = 78.8%

However:

Ranking:
Best-of-4 = 100%

This is not a contradiction.

Because classification asks:
Does the probability assigned to each candidate accurately reflect whether it is correct or incorrect?

Whereas ranking asks:
Did the correct candidate receive a higher score than the incorrect candidates?


